In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()

os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = user_secrets.get_secret("LANGCHAIN_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Zyro-Dynamics-HR-Bot"

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)

In [ ]:
# Change this cell to point to the exact subdirectory where the PDFs live
PDF_DIR = "/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus"

In [ ]:
import os

print("Available datasets:")
print(os.listdir("/kaggle/input"))

In [ ]:
import os

print(os.listdir("/kaggle/input/competitions"))

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file in os.listdir(PDF_DIR):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(PDF_DIR, file))
        documents.extend(loader.load())

print("Pages Loaded:", len(documents))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Using a slightly larger, structured chunk size to prevent document cross-pollution
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False
)
chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Explicitly define the embedding model first so the background runner sees it
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)
# 2. Build the local FAISS index from your document chunks
vectorstore = FAISS.from_documents(chunks, embedding_model)
print("FAISS Index safely built and ready!")

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":4,
        "fetch_k":20
    }
)

In [ ]:
import os 
from langchain_groq import ChatGroq 
from kaggle_secrets import UserSecretsClient 

# Securely load your Groq API Key from Kaggle Secrets 
try: 
    user_secrets = UserSecretsClient() 
    os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY") 
    print("Groq API Key successfully loaded!") 
except Exception as e: 
    print("Error: Make sure GROQ_API_KEY is added and enabled in Add-ons -> Secrets.") 

# Switched to the 8B model to prevent 429 Rate Limits while keeping generation quality high
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.0) 
print("Switched to active Llama 3.1 8B model!")

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are the Zyro Dynamics HR Assistant.

Use ONLY the provided context.

Rules:
1. Answer only from context.
2. Do not invent information.
3. If the answer cannot be found, reply:

I can only answer HR-related questions from Zyro Dynamics policy documents.

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
{
    "context": retriever,
    "question": RunnablePassthrough()
}
| prompt
| llm
| StrOutputParser()
)

In [ ]:
def ask_bot(question):
    # 1. Fetch relevant documents
    docs = retriever.invoke(question)
    if not docs:
        return "I can only answer HR-related questions from Zyro Dynamics policy documents."
    
    # 2. Run the query through the RAG chain
    answer = rag_chain.invoke(question)
    
    # Standardized competitive grader fallback phrase
    fallback_phrase = "I can only answer HR-related questions from Zyro Dynamics policy documents."
    
    # 3. Check if the model or prompt triggered the fallback refusal
    if fallback_phrase.lower() in answer.lower():
        return fallback_phrase
        
    # 4. Clean up source file names (removing duplicates and paths)
    sources = []

for doc in docs:
    source = doc.metadata.get("source", "")
    source = source.split("/")[-1]

    if source not in sources:
        sources.append(source)
return (
    f"{answer}\n\n"
    f"Sources: {', '.join(sources)}"
)

In [ ]:
print("Test 1 (Probation):\n", ask_bot("What is the probation period?"))
print("-" * 50)
print("Test 2 (WFH):\n", ask_bot("What is the work from home policy?"))
print("-" * 50)
print("Test 3 (Notice - Out of Scope Check):\n", ask_bot("How do I bake a chocolate cake?"))
print(ask_bot("Who is the Prime Minister of India?"))
print(ask_bot("Tell me a joke"))
print(ask_bot("What is Bitcoin?"))
print(ask_bot("Who won IPL 2025?"))

In [ ]:
import pandas as pd
from cryptography.fernet import Fernet

# Same secret used by the competition notebook
SUBMISSION_SECRET = b"6Q_EBPtBG-60URcrF6jxNTJSRjy-CtZbJlvp_xf0c_M="
fernet = Fernet(SUBMISSION_SECRET)

question_mapping = {
    "Q01": "What is the probation period for a new employee?",
    "Q02": "How many days of maternity leave are allowed?",
    "Q03": "What is the work from home policy?",
    "Q04": "What is the notice period for an L1 grade employee?",
    "Q05": "Is there a hybrid work option available?",
    "Q06": "What are the rules for emergency leaves?",
    "Q07": "How is the performance review rating calculated?",
    "Q08": "What is covered under the travel and expense policy?",
    "Q09": "What is the company policy on data security for remote work?",
    "Q10": "What is the standard notice period for a senior C-suite executive?",
    "Q11": "Can I bring my pet to the office?",
    "Q12": "What is the stock price of Apple today?",
    "Q13": "How do I bake a chocolate cake?",
    "Q14": "Tell me a joke about programming.",
    "Q15": "Who won the latest football world cup?"
}

STREAMLIT_URL = "https://zyro-hr-chatbot-yxarqo985xdtiz8pkybtrv.streamlit.app/"
LANGSMITH_URL = "https://smith.langchain.com/public/fe19ae2b-bb98-4237-93f4-62c0ef8cde1a/r"

rows = []

for qid, question in question_mapping.items():

    try:
        answer = ask_bot(question)

        if isinstance(answer, dict):
            answer = answer.get("answer", str(answer))

        answer = str(answer)

    except Exception as e:
        answer = f"Error: {str(e)}"

    rows.append({
        "question_id": qid,
        "question_enc": fernet.encrypt(question.encode()).decode(),
        "answer_enc": fernet.encrypt(answer.encode()).decode(),
        "streamlit_link": "https://zyro-hr-chatbot-yxarqo985xdtiz8pkybtrv.streamlit.app/",
        "langsmith_link": "https://smith.langchain.com/public/fe19ae2b-bb98-4237-93f4-62c0ef8cde1a/r"
    })

submission_df = pd.DataFrame(rows)

submission_df.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("✅ submission.csv generated successfully")
print(submission_df.head())

In [ ]:
row=[]
for qid, question in question_mapping.items():

    try:
        answer = ask_bot(question)

        if isinstance(answer, dict):
            answer = answer.get("answer", str(answer))

        answer = str(answer)

    except Exception as e:
        answer = f"Error: {str(e)}"

    rows.append({
        ...
    })

In [ ]:
import os
print(os.listdir('/kaggle/working'))